# Random Forest Prediction of Fluorophore Photophysical Properties

NTU URECA project, independent research.

This notebook contains the full pipeline in order: data loading, exploratory analysis, cleaning, featurization, model evaluation under several train/test splitting strategies, feature engineering, tuning, and model inspection.

---

## Goal

Predict four photophysical properties of organic fluorophores from molecular structure (SMILES):

| Target | Column in raw data | Notes |
|---|---|---|
| Absorption max | `Absorption max (nm)` | |
| Emission max | `Emission max (nm)` | |
| Extinction coefficient | `log(e/mol-1 dm3 cm-1)` | already log10-scaled in the raw data |
| Quantum yield (Φ_F) | `Quantum yield` | bounded 0–1 |

One Random Forest regressor is trained per target.

## Evaluation approach

The main evaluation splits train and test by Murcko scaffold rather than at random. The dataset contains many near-identical molecules, and the same molecule often appears under several solvents. A random split lets these land on both sides of the split, so a model can score well by recognising structures it has already seen. A scaffold split keeps every scaffold on one side only, which tests generalization to new chemical structures. The gap between the two is one of the main results.

## Contents

1. Imports
2. Loading the raw data
3. Renaming columns
4. Sanity checks
5. Exploratory data analysis (one subsection per target)
6. Cleaning: duplicate measurements
7. Featurization: RDKit descriptors and Morgan fingerprints
8. Baseline: random split
9. Scaffold split
10. Hyperparameter tuning with scaffold-grouped CV
11. Frequency-based scaffold split
12. Split variance: scaffold-grouped k-fold CV
13. Solvent features
14. Solvent-held-out split
15. Chemistry-rule features vs. fingerprint bits
16. Cluster-based (Butina) split
17. Final tuned model
18. Model inspection

## Dataset

`chromophore_data.csv`: 20,836 rows, one per (chromophore, solvent) measurement. Columns include the chromophore SMILES, solvent SMILES, the four targets, lifetime, FWHM, molecular weight, and literature reference.

Not every row reports all four targets, and the same molecule can appear several times under different solvents. Both are examined in Section 5.

## 1. Imports

`pandas`/`numpy` for data handling, `matplotlib`/`seaborn` for plots, `rdkit` for SMILES parsing, scaffolds and descriptors, `scikit-learn` for modelling and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')  # suppress RDKit parse warnings

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

print('rdkit, sklearn, pandas all imported OK')

## 2. Load the raw dataset

`df_raw` is kept untouched as a reference copy. All processing happens on `df`.

In [ ]:
DATA_PATH = 'chromophore_data.csv'

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
df = df_raw.copy()  # df_raw stays untouched

print(f'Loaded {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

## 3. Rename columns

The raw column names are shortened for use in code. `df_raw` keeps the originals.

In [ ]:
RENAME_MAP = {
    'Chromophore': 'smiles',
    'Solvent': 'solvent',
    'Absorption max (nm)': 'abs_max',
    'Emission max (nm)': 'emi_max',
    'Quantum yield': 'qy',
    'log(e/mol-1 dm3 cm-1)': 'log_ext_coeff',
    'Molecular weight (g mol-1)': 'mol_weight',
}

df = df.rename(columns=RENAME_MAP)

TARGET_COLS = ['abs_max', 'emi_max', 'log_ext_coeff', 'qy']
ID_COLS = ['smiles', 'solvent']

df[ID_COLS + TARGET_COLS].head()

## 4. Sanity checks

How much data each target has, whether every SMILES parses in RDKit, and how many unique molecules the dataset contains once repeated solvent measurements are accounted for.

In [ ]:
print('Non-null counts per target (out of {:,} rows):'.format(len(df)))
print(df[TARGET_COLS].notna().sum())
print()
print('Unique SMILES (molecules):', df['smiles'].nunique())
print('Unique solvents:', df['solvent'].nunique())

In [ ]:
def parses_ok(smi):
    # MolFromSmiles returns None on failure rather than raising
    try:
        return Chem.MolFromSmiles(smi) is not None
    except Exception:
        return False

parse_mask = df['smiles'].apply(parses_ok)
print(f'SMILES that RDKit can parse: {parse_mask.sum():,} / {len(df):,} ({parse_mask.mean():.1%})')

if (~parse_mask).any():
    print('\nExample unparseable SMILES:')
    display(df.loc[~parse_mask, ['smiles']].head())

## 5. Exploratory data analysis

Each target is examined in turn for coverage, distribution shape and outliers, followed by a short summary of findings.

### 5.1 Absorption max (nm)

In [ ]:
print(f"Missing: {df['abs_max'].isna().sum():,} / {len(df):,} rows ({df['abs_max'].isna().mean():.1%})")
df['abs_max'].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['abs_max'].dropna(), bins=60, ax=axes[0], color='steelblue')
axes[0].set_title('Absorption max — distribution')
axes[0].set_xlabel('Absorption max (nm)')

sns.boxplot(x=df['abs_max'].dropna(), ax=axes[1], color='steelblue')
axes[1].set_title('Absorption max — boxplot (outlier check)')
axes[1].set_xlabel('Absorption max (nm)')

plt.tight_layout()
plt.show()

In [ ]:
# UV-Vis-NIR absorption normally falls within ~200-1000 nm; values outside are checked by hand
low = df[df['abs_max'] < 200][['smiles', 'solvent', 'abs_max']]
high = df[df['abs_max'] > 1000][['smiles', 'solvent', 'abs_max']]

print(f'Below 200 nm: {len(low)} rows')
print(f'Above 1000 nm: {len(high)} rows')
display(pd.concat([low, high]))

**Findings: absorption max**

- Coverage: 17,703 / 20,836 rows (~85%).
- Range 127.7–1055.0 nm, median 397 nm, right-skewed (skew ≈ 1.25). The middle 50% sits between 350 and 480 nm, with a long tail into the near-IR.
- 21 rows fall below 200 nm and 1 above 1000 nm. The sub-200 nm rows are small gas-phase molecules (methane, ethane, propane and similar) that genuinely absorb in the far UV. The >1000 nm row is a large conjugated dye. None are data errors and none were removed.
- No zero or negative values.
- Random Forests are insensitive to skew and isolated outliers, so the target is used untransformed.

### 5.2 Emission max (nm)

The same checks, plus a physical one: emission should occur at a longer wavelength than absorption (a positive Stokes shift).

In [ ]:
print(f"Missing: {df['emi_max'].isna().sum():,} / {len(df):,} rows ({df['emi_max'].isna().mean():.1%})")
df['emi_max'].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['emi_max'].dropna(), bins=60, ax=axes[0], color='darkorange')
axes[0].set_title('Emission max — distribution')
axes[0].set_xlabel('Emission max (nm)')

sns.boxplot(x=df['emi_max'].dropna(), ax=axes[1], color='darkorange')
axes[1].set_title('Emission max — boxplot (outlier check)')
axes[1].set_xlabel('Emission max (nm)')

plt.tight_layout()
plt.show()

In [ ]:
# emission should not occur at a shorter wavelength than absorption (Stokes shift >= 0)
both = df.dropna(subset=['emi_max', 'abs_max']).copy()
both['stokes_shift'] = both['emi_max'] - both['abs_max']

neg_shift = both[both['stokes_shift'] < 0].sort_values('stokes_shift')
print(f"Rows with emission < absorption (negative Stokes shift): {len(neg_shift)} / {len(both)}")
display(neg_shift[['smiles', 'solvent', 'abs_max', 'emi_max', 'stokes_shift']].head(10))

**Findings: emission max**

- Coverage: 18,851 / 20,836 rows (~90%).
- Range 247–1050 nm, median 488 nm, mildly right-skewed (skew ≈ 0.66). The red shift relative to absorption is consistent with the Stokes shift.
- One row above 1000 nm (a long conjugated dye, plausible). None below 200 nm.
- 17 of 16,422 rows (~0.1%) report emission at a shorter wavelength than absorption. 15 of these differ by ≤15 nm and are likely measurement noise. Two differ by -57 and -55 nm (both BODIPY-dimer structures in toluene). They were kept, since there are too few to affect training.
- Used untransformed.

### 5.3 log(extinction coefficient)

Already log10-scaled in the raw data. This is the least complete of the four targets.

In [ ]:
print(f"Missing: {df['log_ext_coeff'].isna().sum():,} / {len(df):,} rows ({df['log_ext_coeff'].isna().mean():.1%})")
df['log_ext_coeff'].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['log_ext_coeff'].dropna(), bins=60, ax=axes[0], color='seagreen')
axes[0].set_title('log(extinction coefficient) — distribution')
axes[0].set_xlabel('log10(ε / mol⁻¹ dm³ cm⁻¹)')

sns.boxplot(x=df['log_ext_coeff'].dropna(), ax=axes[1], color='seagreen')
axes[1].set_title('log(extinction coefficient) — boxplot (outlier check)')
axes[1].set_xlabel('log10(ε / mol⁻¹ dm³ cm⁻¹)')

plt.tight_layout()
plt.show()

In [ ]:
# molar extinction coefficients of organic dyes rarely exceed ~10^6 (log10 ~ 6)
high = df[df['log_ext_coeff'] > 6][['smiles', 'solvent', 'log_ext_coeff', 'mol_weight', 'Reference']]
print(f'Rows above log10(ε) = 6: {len(high)}')
display(high)

In [ ]:
# 10.4 and 9.0 come from one paper whose other near-identical compounds sit at 5.5-6.0;
# treated as data-entry errors (originals remain in df_raw)
error_mask = df['log_ext_coeff'] > 7
print(f'Setting {error_mask.sum()} likely data-error rows to NaN in log_ext_coeff')
df.loc[error_mask, 'log_ext_coeff'] = np.nan

**Findings: log(extinction coefficient)**

- Coverage: 8,300 / 20,836 rows (~40%), the lowest of the four targets.
- Range 1.08–10.4 before cleaning, median 4.46, left-skewed (skew ≈ -1.40). Most values lie between 4.1 and 4.7, i.e. ε of roughly 10⁴–10⁵.
- Four rows exceed log10(ε) = 6. Two of them (10.4 and 9.0) come from the same paper and the same family of near-identical molecules, all other members of which sit around 5.5–6.0. A molar extinction coefficient of 10⁹–10¹⁰ is physically implausible for an organic dye, so these two were treated as data-entry errors and set to NaN. The remaining two (6.02, 6.11) are plausible for strongly absorbing dyes and were kept.
- This target has the smallest training set, which limits how precisely its scaffold-split performance can be measured.

### 5.4 Quantum yield (Φ_F)

Bounded between 0 and 1 by definition. The checks focus on whether the data respects that bound and on the shape of the distribution near the edges.

In [ ]:
print(f"Missing: {df['qy'].isna().sum():,} / {len(df):,} rows ({df['qy'].isna().mean():.1%})")
df['qy'].describe()

In [ ]:
qy = df['qy']
print('Values < 0:', (qy < 0).sum())
print('Values > 1:', (qy > 1).sum())
print('Values == 0:', (qy == 0).sum())
print('Values < 0.01:', (qy < 0.01).sum())
print('Values > 0.99:', (qy > 0.99).sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['qy'].dropna(), bins=50, ax=axes[0], color='mediumorchid')
axes[0].set_title('Quantum yield — distribution')
axes[0].set_xlabel('Quantum yield (Φ_F)')

sns.boxplot(x=df['qy'].dropna(), ax=axes[1], color='mediumorchid')
axes[1].set_title('Quantum yield — boxplot')
axes[1].set_xlabel('Quantum yield (Φ_F)')

plt.tight_layout()
plt.show()

**Findings: quantum yield**

- Coverage: 13,978 / 20,836 rows (~67%).
- All values lie within [0, 1].
- Median 0.27, mean 0.35, skew ≈ 0.57. The skew understates the problem: 150 rows are exactly 0 and 1,273 (~9%) are below 0.01, a sharp spike of non-emissive molecules. A smaller cluster sits near 1 (148 rows above 0.99). The distribution is closer to bimodal with a large mass at zero than to a smooth skewed curve.
- A zero-inflated target is hard for any regressor, since the model has to learn a sharp boundary between non-emissive and emissive molecules rather than a smooth function of the features. This target is expected to perform worst.

### 5.5 Modelling decisions from the EDA

- One Random Forest per target, no multi-output model.
- Absorption and emission max: no transform. Skew is mild (1.25, 0.66) and all outliers checked were real chemistry.
- log(extinction coefficient): used on its existing log10 scale. Two data-entry errors (10.4, 9.0) set to NaN (Section 5.3).
- Quantum yield: modelled on the raw [0, 1] scale for the baseline, despite the zero inflation, so that all four targets use the same recipe. A logit or arcsine-square-root transform and a two-stage classify-then-regress model are candidate improvements to compare against this baseline.
- No capping or winsorizing. Random Forests isolate extreme points in their own leaves rather than letting them distort a global fit.
- Derived quantities such as the Stokes shift are used for analysis only. They are computed from a second target, so using them as inputs would leak target information.

## 6. Cleaning: duplicate measurements

Some (molecule, solvent) pairs appear more than once because different papers measured the same compound.

In [ ]:
dup_mask = df.duplicated(subset=['smiles', 'solvent'], keep=False)
n_dup_rows = dup_mask.sum()
n_dup_groups = df[dup_mask].groupby(['smiles', 'solvent']).ngroups

print(f'Rows involved in a duplicate (smiles, solvent) pair: {n_dup_rows} / {len(df)}')
print(f'Number of distinct duplicated groups: {n_dup_groups}')

spread = (df[dup_mask].groupby(['smiles', 'solvent'])['abs_max']
          .agg(lambda x: x.dropna().max() - x.dropna().min() if x.notna().sum() > 1 else np.nan))
print(f"\nAbsorption max spread within duplicate groups (where measured twice):")
print(spread.dropna().describe())
print(f"Groups disagreeing by >10 nm: {(spread.dropna() > 10).sum()} / {spread.dropna().shape[0]}")

In [ ]:
# median is robust to a single disagreeing paper; references are kept for traceability
agg_dict = {col: 'median' for col in TARGET_COLS}
agg_dict['mol_weight'] = 'median'
agg_dict['Reference'] = lambda refs: '; '.join(sorted(set(refs.dropna().astype(str))))

df = df.groupby(['smiles', 'solvent'], as_index=False).agg(agg_dict)

print(f'Rows before dedup: 20,836  ->  after dedup: {len(df):,}')
print(f'Unique molecules: {df["smiles"].nunique():,}')
print()
print('Non-null counts per target after dedup:')
print(df[TARGET_COLS].notna().sum())

**Cleaning summary**

- 618 rows (~3%) share a (SMILES, solvent) pair with at least one other row, across 304 groups. Most groups agree closely; 22 disagree by more than 10 nm on absorption max. Duplicates were collapsed to one row per pair using the median of each target, giving 20,836 → 20,522 rows. The number of unique molecules (6,865) is unchanged.
- Two log(extinction coefficient) errors were already set to NaN in Section 5.3.
- All SMILES parse (Section 4), so no rows were dropped on that basis.

`df` is now the cleaned dataset. `df_raw` keeps the original 20,836 rows.

## 7. Featurization

Two kinds of features are computed from each chromophore SMILES:

- 11 RDKit physicochemical descriptors (molecular weight, LogP, TPSA, H-bond donors and acceptors, ring counts, and others). These are interpretable and useful for feature importance.
- A Morgan fingerprint (ECFP, radius 2, 512 bits), which encodes local substructure patterns.

Features depend only on the molecule, not the solvent, so they are computed once per unique SMILES (6,865) and joined back onto the 20,522 rows.

In [ ]:
from rdkit.Chem import rdMolDescriptors

DESCRIPTOR_FNS = {
    'MolWt': Descriptors.MolWt,
    'MolLogP': Descriptors.MolLogP,
    'TPSA': Descriptors.TPSA,
    'NumHDonors': Descriptors.NumHDonors,
    'NumHAcceptors': Descriptors.NumHAcceptors,
    'NumRotatableBonds': Descriptors.NumRotatableBonds,
    'RingCount': Descriptors.RingCount,
    'NumAromaticRings': Descriptors.NumAromaticRings,
    'FractionCSP3': Descriptors.FractionCSP3,
    'HeavyAtomCount': Descriptors.HeavyAtomCount,
    'NumValenceElectrons': Descriptors.NumValenceElectrons,
}
FP_RADIUS = 2
FP_BITS = 512

unique_smiles = df['smiles'].unique()
print(f'Featurizing {len(unique_smiles):,} unique molecules...')

feature_rows = []
for smi in unique_smiles:
    mol = Chem.MolFromSmiles(smi)
    row = {name: fn(mol) for name, fn in DESCRIPTOR_FNS.items()}
    fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=FP_RADIUS, nBits=FP_BITS)
    row.update({f'fp_{i}': int(bit) for i, bit in enumerate(fp)})
    row['smiles'] = smi
    feature_rows.append(row)

feat_df = pd.DataFrame(feature_rows)
FEATURE_COLS = list(DESCRIPTOR_FNS.keys()) + [f'fp_{i}' for i in range(FP_BITS)]
print(f'Feature matrix: {feat_df.shape[0]:,} molecules x {len(FEATURE_COLS)} features')

In [ ]:
df = df.merge(feat_df, on='smiles', how='left')
df[['smiles', 'solvent'] + list(DESCRIPTOR_FNS.keys())].head()

## 8. Baseline: random split

A random 80/20 split with a default Random Forest (200 trees), one model per target. This serves as a reference point for the scaffold-split results that follow.

In [ ]:
random_split_results = {}

for target in TARGET_COLS:
    sub = df.dropna(subset=[target])
    X = sub[FEATURE_COLS]
    y = sub[target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    pred = rf.predict(X_test)

    random_split_results[target] = {
        'n_train': len(X_train),
        'n_test': len(X_test),
        'R2': r2_score(y_test, pred),
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': mean_squared_error(y_test, pred) ** 0.5,
    }

pd.DataFrame(random_split_results).T

**Random-split results**

| Target | n train | n test | R² | MAE | RMSE |
|---|---|---|---|---|---|
| Absorption max | 13,957 | 3,490 | 0.931 | 14.2 nm | 28.1 nm |
| Emission max | 14,852 | 3,714 | 0.845 | 24.4 nm | 36.6 nm |
| log(ext coeff) | 6,551 | 1,638 | 0.826 | 0.139 | 0.239 |
| Quantum yield | 10,998 | 2,750 | 0.546 | 0.145 | 0.206 |

The ranking matches the EDA: the wavelength targets are easiest and quantum yield is hardest.

These scores are likely optimistic. The same molecule appears under several solvents, so a random split can place one measurement of a compound in training and another in test. Fingerprints are computed from the molecule alone, so those two rows have identical inputs and the model has effectively already seen the test molecule. The scaffold split in Section 9 removes this leakage path.

## 9. Scaffold split

Same model, features and hyperparameters as Section 8. Only the split changes: molecules are grouped by Murcko scaffold and whole scaffold groups are assigned to train or test, so no scaffold appears on both sides.

In [ ]:
# acyclic molecules get an empty scaffold string
scaffold_map = {
    smi: MurckoScaffold.MurckoScaffoldSmiles(mol=Chem.MolFromSmiles(smi))
    for smi in df['smiles'].unique()
}
df['scaffold'] = df['smiles'].map(scaffold_map)

print(f"Unique scaffolds: {df['scaffold'].nunique():,} (from {df['smiles'].nunique():,} unique molecules)")
print(f"Rows with no ring system (empty scaffold): {(df['scaffold'] == '').sum()}")
df['scaffold'].value_counts().head()

In [ ]:
# target ~80/20 by row count, not scaffold count: group sizes range from 1 to 400+
scaffold_sizes = df.groupby('scaffold').size().sample(frac=1, random_state=42)
cum_rows = scaffold_sizes.cumsum()
test_scaffolds = set(cum_rows[cum_rows <= 0.2 * len(df)].index)

df['split'] = np.where(df['scaffold'].isin(test_scaffolds), 'test', 'train')

print(df['split'].value_counts())
print(f"Distinct scaffolds in train: {df.loc[df['split']=='train','scaffold'].nunique():,}")
print(f"Distinct scaffolds in test:  {df.loc[df['split']=='test','scaffold'].nunique():,}")

overlap = set(df.loc[df['split']=='train','scaffold']) & set(df.loc[df['split']=='test','scaffold'])
print(f"Scaffolds present in both (must be 0): {len(overlap)}")

In [ ]:
scaffold_split_results = {}

for target in TARGET_COLS:
    sub = df.dropna(subset=[target])
    train_sub = sub[sub['split'] == 'train']
    test_sub = sub[sub['split'] == 'test']

    X_train, y_train = train_sub[FEATURE_COLS], train_sub[target]
    X_test, y_test = test_sub[FEATURE_COLS], test_sub[target]

    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    pred = rf.predict(X_test)

    scaffold_split_results[target] = {
        'n_train': len(X_train),
        'n_test': len(X_test),
        'R2': r2_score(y_test, pred),
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': mean_squared_error(y_test, pred) ** 0.5,
    }

pd.DataFrame(scaffold_split_results).T

**Scaffold-split results vs. random split**

| Target | R² (random) | R² (scaffold) | Change | MAE (random) | MAE (scaffold) |
|---|---|---|---|---|---|
| Absorption max | 0.931 | 0.137 | -0.79 | 14.2 nm | 63.9 nm |
| Emission max | 0.845 | 0.651 | -0.19 | 24.4 nm | 41.0 nm |
| log(ext coeff) | 0.826 | 0.599 | -0.23 | 0.139 | 0.238 |
| Quantum yield | 0.546 | 0.302 | -0.24 | 0.145 | 0.213 |

3,524 unique scaffolds from 6,865 molecules. Train: 2,817 scaffolds / 16,426 rows. Test: 707 scaffolds / 4,096 rows. No scaffold overlap.

Every target drops when moving to the scaffold split. Absorption max, the best target under the random split, drops furthest, from R² 0.93 to 0.14, while the other three fall by 0.19–0.24. Taken at face value this suggests absorption max relies most heavily on recognising near-identical molecules. Sections 11 and 12 show that this particular number is largely an artefact of a single unlucky split.

## 10. Hyperparameter tuning with scaffold-grouped CV

Tuning uses the scaffold split, since that is the evaluation of interest.

Grid: `n_estimators` ∈ {100, 200, 300}, `max_depth` ∈ {None, 10, 20}, `max_features` ∈ {'sqrt', 0.3}, `min_samples_leaf` ∈ {1, 4}, 36 combinations.

Cross-validation inside the search uses 5-fold `GroupKFold` grouped by scaffold, on the training set only. Plain `KFold` would let one scaffold's rows fall into both a training fold and a validation fold, which would bias hyperparameter selection toward memorisation. One search per target; total runtime about 27 minutes.

In [ ]:
from sklearn.model_selection import GroupKFold, GridSearchCV

PARAM_GRID = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'max_features': ['sqrt', 0.3],
    'min_samples_leaf': [1, 4],
}

tuned_results = {}

for target in TARGET_COLS:
    sub = df.dropna(subset=[target])
    train_sub = sub[sub['split'] == 'train']
    test_sub = sub[sub['split'] == 'test']

    X_train, y_train = train_sub[FEATURE_COLS], train_sub[target]
    X_test, y_test = test_sub[FEATURE_COLS], test_sub[target]
    groups_train = train_sub['scaffold']

    # a generator can't be pickled for parallel workers
    gkf = GroupKFold(n_splits=5)
    cv_splits = list(gkf.split(X_train, y_train, groups=groups_train))

    rf = RandomForestRegressor(random_state=42, n_jobs=1)  # parallelism at the search level
    search = GridSearchCV(rf, PARAM_GRID, cv=cv_splits, scoring='r2', n_jobs=-1)
    search.fit(X_train, y_train)

    best_model = search.best_estimator_
    pred = best_model.predict(X_test)

    tuned_results[target] = {
        'n_train': len(X_train),
        'n_test': len(X_test),
        'best_params': search.best_params_,
        'cv_best_score': search.best_score_,
        'test_R2': r2_score(y_test, pred),
        'test_MAE': mean_absolute_error(y_test, pred),
        'test_RMSE': mean_squared_error(y_test, pred) ** 0.5,
    }
    print(f'[{target}] best_params={search.best_params_}  CV R2={search.best_score_:.3f}  test R2={tuned_results[target]["test_R2"]:.3f}')

pd.DataFrame(tuned_results).T

**Tuning results**

| Target | Best params | CV R² (train, scaffold-grouped) | Test R² (untuned → tuned) | Test MAE (untuned → tuned) |
|---|---|---|---|---|
| Absorption max | n_estimators=300, max_depth=None, max_features=0.3, min_samples_leaf=1 | 0.744 | 0.137 → 0.184 | 63.9 → 60.9 nm |
| Emission max | n_estimators=300, max_depth=None, max_features=0.3, min_samples_leaf=1 | 0.637 | 0.651 → 0.666 | 41.0 → 40.7 nm |
| log(ext coeff) | n_estimators=100, max_depth=None, max_features='sqrt', min_samples_leaf=1 | 0.447 | 0.599 → 0.554 | 0.238 → 0.251 |
| Quantum yield | n_estimators=200, max_depth=None, max_features='sqrt', min_samples_leaf=1 | 0.240 | 0.302 → 0.300 | 0.213 → 0.218 |

Tuning has little effect. The largest gain is +0.05 R² (absorption max), log(ext coeff) gets slightly worse on the test set, and quantum yield is unchanged. Depth, tree count and leaf size control how closely the model fits its training data. They cannot create signal that transfers to unseen scaffolds, so the scaffold gap is a limitation of the features and data rather than of the hyperparameters.

Absorption max shows a large gap between its CV score during the search (0.744) and its held-out test score (0.184). The other three targets have CV and test scores within about 0.15 of each other. Either the training scaffolds are unusually predictive of each other for this target, or the specific test scaffolds in this split are unusually hard. Sections 11 and 12 distinguish between these explanations.

## 11. Frequency-based scaffold split

In Section 9 scaffold groups were assigned to train or test at random. Here they are sorted by size and the largest groups fill the training set until it holds ~80% of rows, so the test set contains only the rarest scaffolds. Same untuned model as Section 9.

In [ ]:
# largest scaffold groups fill train; the rarest go to test
scaffold_sizes_desc = df.groupby('scaffold').size().sort_values(ascending=False)
cum_rows_desc = scaffold_sizes_desc.cumsum()
train_scaffolds_freq = set(cum_rows_desc[cum_rows_desc <= 0.8 * len(df)].index)

df['split_freq'] = np.where(df['scaffold'].isin(train_scaffolds_freq), 'train', 'test')

print(df['split_freq'].value_counts())
print(f"Distinct scaffolds in train: {df.loc[df['split_freq']=='train','scaffold'].nunique():,}")
print(f"Distinct scaffolds in test:  {df.loc[df['split_freq']=='test','scaffold'].nunique():,}")

overlap_freq = set(df.loc[df['split_freq']=='train','scaffold']) & set(df.loc[df['split_freq']=='test','scaffold'])
print(f"Scaffolds present in both (must be 0): {len(overlap_freq)}")

In [ ]:
freq_split_results = {}

for target in TARGET_COLS:
    sub = df.dropna(subset=[target])
    train_sub = sub[sub['split_freq'] == 'train']
    test_sub = sub[sub['split_freq'] == 'test']

    X_train, y_train = train_sub[FEATURE_COLS], train_sub[target]
    X_test, y_test = test_sub[FEATURE_COLS], test_sub[target]

    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    pred = rf.predict(X_test)

    freq_split_results[target] = {
        'n_train': len(X_train),
        'n_test': len(X_test),
        'R2': r2_score(y_test, pred),
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': mean_squared_error(y_test, pred) ** 0.5,
    }

pd.DataFrame(freq_split_results).T

**Frequency-split results vs. random scaffold split (Section 9)**

| Target | R² (random scaffold) | R² (frequency scaffold) | Change |
|---|---|---|---|
| Absorption max | 0.137 | 0.584 | +0.447 |
| Emission max | 0.651 | 0.548 | -0.103 |
| log(ext coeff) | 0.599 | 0.337 | -0.262 |
| Quantum yield | 0.302 | 0.102 | -0.200 |

Train: the 1,188 most common scaffolds (16,416 rows). Test: the 2,336 rarest (4,106 rows). No overlap.

Absorption max rises from 0.137 to 0.584 when every common scaffold is forced into training. This points to the Section 9 score being driven by which scaffolds happened to land in that test set. If a few large, structurally unusual scaffold groups were assigned to test by chance, their errors alone could pull R² down sharply.

The other three targets move the opposite way. log(ext coeff) and quantum yield have the least data (8,189 and 13,748 rows), so testing only on rare, near-singleton scaffolds leaves little related structure in training to draw on.

A single scaffold split can therefore shift a target's R² by more than 0.4 depending on which scaffolds land in test. Section 12 measures that variance directly.

## 12. Split variance: scaffold-grouped k-fold CV

**Hypothesis.** Absorption max's Section 9 score (R² 0.137) reflects the particular scaffolds assigned to that test set rather than a general failure to generalize. If so, different scaffold splits should give very different scores for this target.

**Test.** 5-fold `GroupKFold` grouped by scaffold over the full dataset, for all four targets. Each fold holds out a different fifth of the scaffolds, giving five independent estimates per target. If the hypothesis holds, absorption max should show much larger fold-to-fold variation than the other targets and a mean well above 0.137. The untuned model (200 trees) is used so that the comparison reflects split variance only.

In [ ]:
kfold_r2 = {}

for target in TARGET_COLS:
    sub = df.dropna(subset=[target]).reset_index(drop=True)
    X, y, groups = sub[FEATURE_COLS], sub[target], sub['scaffold']

    gkf = GroupKFold(n_splits=5)
    fold_scores = []
    for train_idx, test_idx in gkf.split(X, y, groups=groups):
        rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
        rf.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = rf.predict(X.iloc[test_idx])
        fold_scores.append(r2_score(y.iloc[test_idx], pred))

    kfold_r2[target] = fold_scores
    print(f'{target}: {[round(s, 3) for s in fold_scores]}  '
          f'mean={np.mean(fold_scores):.3f}  std={np.std(fold_scores):.3f}')

**Results**

| Target | Fold scores | Mean R² | Std R² |
|---|---|---|---|
| Absorption max | 0.713, 0.334, 0.725, 0.706, 0.729 | 0.641 | 0.154 |
| Emission max | 0.629, 0.618, 0.616, 0.643, 0.625 | 0.626 | 0.010 |
| log(ext coeff) | 0.508, 0.392, 0.541, 0.514, 0.312 | 0.454 | 0.088 |
| Quantum yield | 0.265, 0.268, 0.183, 0.297, 0.195 | 0.242 | 0.044 |

The hypothesis is supported. Absorption max's standard deviation (0.154) is 1.75–15 times that of the other targets. Four of its folds fall between 0.706 and 0.729; the fifth is 0.334, still more than double the Section 9 score. The Section 9 split was an extreme draw, and absorption max's typical scaffold-split R² is about 0.64, comparable to emission max.

Emission max is the most stable target (std 0.010): its R² of ~0.62–0.65 holds regardless of which scaffolds are held out. log(ext coeff) and quantum yield show moderate variance, consistent with their smaller datasets.

From this section on, scaffold-based results are reported as k-fold means and standard deviations. The single-split numbers in Sections 9–11 are individual samples from this distribution.

## 13. Solvent features

The features so far describe only the chromophore, so the model cannot distinguish the same molecule measured in two different solvents. The `Solvent` column is itself SMILES (`O` = water, `ClCCl` = dichloromethane, `Cc1ccccc1` = toluene), so solvents can be featurized the same way. Five polarity-related descriptors are used: molecular weight, LogP, TPSA, and H-bond donor and acceptor counts.

20 rows have `Solvent = 'gas'`, which is not valid SMILES. These are the gas-phase far-UV measurements from Section 5.1. They receive an `is_gas_phase` flag with the other solvent descriptors set to zero.

The extended feature set is `FEATURE_COLS_SOLV`. Sections 8–12 continue to use the molecule-only `FEATURE_COLS`.

In [ ]:
SOLVENT_DESCRIPTOR_FNS = {
    'solvent_MolWt': Descriptors.MolWt,
    'solvent_MolLogP': Descriptors.MolLogP,
    'solvent_TPSA': Descriptors.TPSA,
    'solvent_NumHDonors': Descriptors.NumHDonors,
    'solvent_NumHAcceptors': Descriptors.NumHAcceptors,
}

unique_solvents = df['solvent'].unique()
solvent_rows = []
for solv in unique_solvents:
    mol = Chem.MolFromSmiles(solv)
    if mol is None:
        # 'gas' is not SMILES: gas-phase measurement, no solvent
        row = {name: 0.0 for name in SOLVENT_DESCRIPTOR_FNS}
        row['solvent_is_gas_phase'] = 1
    else:
        row = {name: fn(mol) for name, fn in SOLVENT_DESCRIPTOR_FNS.items()}
        row['solvent_is_gas_phase'] = 0
    row['solvent'] = solv
    solvent_rows.append(row)

solvent_feat_df = pd.DataFrame(solvent_rows)
SOLVENT_FEATURE_COLS = list(SOLVENT_DESCRIPTOR_FNS.keys()) + ['solvent_is_gas_phase']

df = df.merge(solvent_feat_df, on='solvent', how='left')
FEATURE_COLS_SOLV = FEATURE_COLS + SOLVENT_FEATURE_COLS

print(f'Solvent features computed for {len(unique_solvents):,} unique solvents')
print(f'Molecule-only features: {len(FEATURE_COLS)}  ->  with solvent features: {len(FEATURE_COLS_SOLV)}')
df[['smiles', 'solvent'] + SOLVENT_FEATURE_COLS].drop_duplicates('solvent').head()

## 14. Solvent-held-out split

This split tests generalization to new solvent environments rather than new molecules. All rows for a given solvent go to the same side. The same molecule usually appears on both sides in different solvents. That is intended: the solvent descriptors are now inputs, so the model has to use them to predict the solvent-dependent shift.

Evaluated with solvent-grouped 5-fold CV.

In [ ]:
kfold_solvent_r2 = {}

for target in TARGET_COLS:
    sub = df.dropna(subset=[target]).reset_index(drop=True)
    X, y, groups = sub[FEATURE_COLS_SOLV], sub[target], sub['solvent']

    gkf = GroupKFold(n_splits=5)
    fold_scores = []
    for train_idx, test_idx in gkf.split(X, y, groups=groups):
        rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
        rf.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = rf.predict(X.iloc[test_idx])
        fold_scores.append(r2_score(y.iloc[test_idx], pred))

    kfold_solvent_r2[target] = fold_scores
    print(f'{target}: {[round(s, 3) for s in fold_scores]}  '
          f'mean={np.mean(fold_scores):.3f}  std={np.std(fold_scores):.3f}')

**Results vs. scaffold-split k-fold means (Section 12)**

| Target | Fold scores (solvent held out) | Mean R² | Std R² | Mean R² (scaffold) |
|---|---|---|---|---|
| Absorption max | 0.822, 0.940, 0.849, 0.934, 0.905 | 0.890 | 0.047 | 0.641 |
| Emission max | 0.755, 0.872, 0.783, 0.860, 0.850 | 0.824 | 0.046 | 0.626 |
| log(ext coeff) | 0.437, 0.791, 0.701, 0.783, 0.566 | 0.656 | 0.136 | 0.454 |
| Quantum yield | 0.448, 0.525, 0.546, 0.592, 0.584 | 0.539 | 0.052 | 0.242 |

1,363 unique solvents; dichloromethane, acetonitrile and toluene are the largest groups.

Every target generalizes much better to new solvents than to new scaffolds. The main source of difficulty in this problem is chemical structure, not solvent environment. This is physically reasonable: solvent effects on absorption and emission are typically shifts of a few to a few tens of nm, while a new scaffold changes the underlying electronic structure.

Quantum yield reaches only R² 0.54 even on this easier task, consistent with the zero-inflation problem from Section 5.4 being independent of the generalization axis.

## 15. Chemistry-rule features vs. fingerprint bits

**Hypothesis.** A Morgan fingerprint records which exact local substructures a molecule contains. That suits recognising structures seen in training, but a new scaffold produces mostly unfamiliar bit patterns. A feature built on a physical mechanism, such as the presence of a conjugated donor–acceptor system, describes something that should still apply to an unseen scaffold. If so, a small set of mechanism-based features should generalize across the scaffold split at least as well as 512 fingerprint bits.

**Rule features** (20, added to the 11 existing descriptors), computed with SMARTS patterns and graph traversal:
- Donor subtypes: primary, secondary and tertiary amine, phenolic –OH, aryl ether. Electron donors push density into the π-system and red-shift absorption.
- Acceptor subtypes: nitrile, nitro, carbonyl, CF₃, sulfonyl. Electron acceptors have the complementary effect.
- `tict_rotor`: a dialkylamino group on an aromatic ring through a rotatable bond. Twisting in the excited state opens a non-radiative decay path and lowers Φ.
- `halogen_heavy`, `halogen_any`: the heavy-atom effect increases intersystem crossing and lowers Φ.
- `donor_total`, `acceptor_total`, `has_push_pull`, `donor_acceptor_product`: aggregate and interaction terms.
- `conjugation_size`: atom count of the largest connected conjugated system (union-find over conjugated bonds). Longer conjugation narrows the HOMO–LUMO gap.
- `formal_charge`, `ring_fraction`: ionic character and rigidity. Rigid molecules lose less energy to vibration and tend to have higher Φ.

Each SMARTS pattern was checked against reference molecules. DMABN, the standard TICT dye, triggers the tertiary-amine donor, nitrile acceptor and TICT-rotor flags as expected.

**Test.** Three feature sets, evaluated with the same scaffold-grouped 5-fold CV and untuned model as Section 12:
- `fingerprint_only`: 11 descriptors + 512 fingerprint bits (523 features)
- `rule_only`: 11 descriptors + 20 rule features, no fingerprint (31 features)
- `combined`: all of the above (543 features)

In [ ]:
SMARTS_PATTERNS = {
    'donor_NH2': '[NX3H2][#6]',                       # primary amine
    'donor_NH1': '[NX3H1]([#6])[#6]',                 # secondary amine
    'donor_NH0': '[NX3H0]([#6])([#6])[#6]',           # tertiary amine
    'donor_OH_arom': '[OX2H][c]',                      # phenol
    'donor_OR_arom': '[OX2]([#6])[c]',                 # aryl ether/alkoxy
    'acceptor_nitrile': 'C#N',
    'acceptor_nitro': '[N+](=O)[O-]',
    'acceptor_carbonyl': '[CX3]=[OX1]',
    'acceptor_CF3': 'C(F)(F)F',
    'acceptor_sulfonyl': 'S(=O)(=O)',
    'tict_rotor': '[NX3]([#6])([#6])!@[c]',           # dialkylamino on aromatic, rotatable bond
    'halogen_heavy': '[Br,I]',
    'halogen_any': '[F,Cl,Br,I]',
}
COMPILED_SMARTS = {name: Chem.MolFromSmarts(s) for name, s in SMARTS_PATTERNS.items()}
assert all(p is not None for p in COMPILED_SMARTS.values()), 'a SMARTS pattern failed to compile'

# check patterns against reference molecules
test_cases = {
    'DMABN (classic TICT dye)': 'CN(C)c1ccc(C#N)cc1',   # should hit donor_NH0, acceptor_nitrile, tict_rotor
    'nitrobenzene': 'c1ccccc1[N+](=O)[O-]',
    'phenol': 'c1ccccc1O',
    'bromobenzene': 'c1ccccc1Br',
}
for label, smi in test_cases.items():
    mol = Chem.MolFromSmiles(smi)
    hits = {name: len(mol.GetSubstructMatches(patt)) for name, patt in COMPILED_SMARTS.items()
            if len(mol.GetSubstructMatches(patt)) > 0}
    print(f'{label}: {hits}')

In [ ]:
def largest_conjugated_system(mol):
    # union-find over atoms connected by a conjugated bond; return the size of the biggest cluster
    n = mol.GetNumAtoms()
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    for bond in mol.GetBonds():
        if bond.GetIsConjugated():
            union(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())
    sizes = {}
    for atom in mol.GetAtoms():
        root = find(atom.GetIdx())
        sizes[root] = sizes.get(root, 0) + 1
    return max(sizes.values()) if sizes else 0


rule_rows = []
for smi in unique_smiles:
    mol = Chem.MolFromSmiles(smi)
    smarts_counts = {name: len(mol.GetSubstructMatches(patt)) for name, patt in COMPILED_SMARTS.items()}

    row = dict(smarts_counts)
    row['donor_total'] = sum(smarts_counts[k] for k in ['donor_NH2', 'donor_NH1', 'donor_NH0', 'donor_OH_arom', 'donor_OR_arom'])
    row['acceptor_total'] = sum(smarts_counts[k] for k in ['acceptor_nitrile', 'acceptor_nitro', 'acceptor_carbonyl', 'acceptor_CF3', 'acceptor_sulfonyl'])
    row['has_push_pull'] = int(row['donor_total'] > 0 and row['acceptor_total'] > 0)
    row['donor_acceptor_product'] = row['donor_total'] * row['acceptor_total']
    row['conjugation_size'] = largest_conjugated_system(mol)
    row['formal_charge'] = Chem.GetFormalCharge(mol)
    n_ring_atoms = sum(1 for atom in mol.GetAtoms() if atom.IsInRing())
    row['ring_fraction'] = n_ring_atoms / mol.GetNumHeavyAtoms() if mol.GetNumHeavyAtoms() > 0 else 0.0
    row['smiles'] = smi
    rule_rows.append(row)

rule_feat_df = pd.DataFrame(rule_rows)
df = df.merge(rule_feat_df, on='smiles', how='left')

RULE_FEATURE_COLS = (list(DESCRIPTOR_FNS.keys()) + list(SMARTS_PATTERNS.keys()) +
                      ['donor_total', 'acceptor_total', 'has_push_pull', 'donor_acceptor_product',
                       'conjugation_size', 'formal_charge', 'ring_fraction'])

FEATURE_SETS = {
    'fingerprint_only': FEATURE_COLS,
    'rule_only': RULE_FEATURE_COLS,
    'combined': FEATURE_COLS + [c for c in RULE_FEATURE_COLS if c not in DESCRIPTOR_FNS],
}
print('Feature set sizes:', {k: len(v) for k, v in FEATURE_SETS.items()})

In [ ]:
h2_results = {}

for target in TARGET_COLS:
    sub = df.dropna(subset=[target]).reset_index(drop=True)
    groups = sub['scaffold']
    h2_results[target] = {}

    for set_name, cols in FEATURE_SETS.items():
        X, y = sub[cols], sub[target]
        gkf = GroupKFold(n_splits=5)
        fold_scores = []
        importances_accum = None

        for train_idx, test_idx in gkf.split(X, y, groups=groups):
            rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
            rf.fit(X.iloc[train_idx], y.iloc[train_idx])
            pred = rf.predict(X.iloc[test_idx])
            fold_scores.append(r2_score(y.iloc[test_idx], pred))
            importances_accum = rf.feature_importances_.copy() if importances_accum is None else importances_accum + rf.feature_importances_

        mean_importances = importances_accum / 5
        top_features = sorted(zip(cols, mean_importances), key=lambda t: -t[1])[:6]

        h2_results[target][set_name] = {
            'mean_r2': np.mean(fold_scores),
            'std_r2': np.std(fold_scores),
            'n_features': len(cols),
            'top_features': top_features,
        }
        print(f'[{target} / {set_name}] mean R2={np.mean(fold_scores):.3f} std={np.std(fold_scores):.3f} '
              f'(n_features={len(cols)})')

**Results**

| Target | fingerprint_only (523) | rule_only (31) | combined (543) |
|---|---|---|---|
| Absorption max | 0.641 | 0.481 | 0.659 |
| Emission max | 0.626 | 0.527 | 0.650 |
| log(ext coeff) | 0.454 | 0.204 | 0.476 |
| Quantum yield | 0.242 | 0.093 | 0.244 |

The strong form of the hypothesis is not supported. Rule-only features lose to fingerprint-only on every target, by as much as 0.25 R² for log(ext coeff). Thirty-one mechanism-based features do not carry enough information to replace the fingerprint.

The combined set beats fingerprint-only on every target, by 0.018–0.024 R² for three targets and essentially level for quantum yield. The rule features add information the fingerprint lacks, even when outnumbered 512 to 20 in the pool of features available at each split.

`conjugation_size` ranks in the top two or three features for nearly every target and feature set. It is outranked only by a single fingerprint bit, `fp_463`, which is the most important feature for absorption max, emission max and log(ext coeff). The model's reliance on conjugation length is consistent with the expected physics: longer conjugation narrows the energy gap and red-shifts absorption and emission.

Other patterns in the importances:
- For quantum yield, the top rule-only features are `MolLogP`, `TPSA`, `ring_fraction` and `FractionCSP3`, i.e. polarity and rigidity.
- `donor_NH0` (tertiary amine donor) ranks highly for emission max but not absorption max, consistent with TICT being an excited-state effect.
- `halogen_any` appears among the top features for absorption max, though the mechanism is less clear.

`fp_463` is examined in Section 18.

## 16. Cluster-based (Butina) split

Scaffold splitting groups molecules by their exact Murcko core, so two molecules with different cores but similar overall structure can still land on opposite sides. Cluster-based splitting groups by overall similarity instead.

Butina clustering on pairwise Tanimoto distance between Morgan fingerprints, with a distance cutoff of 0.4 (similarity 0.6). Every member of a cluster is within the cutoff of the cluster centroid. Evaluated with cluster-grouped 5-fold CV.

In [ ]:
from rdkit import DataStructs
from rdkit.ML.Cluster import Butina

cluster_fps = [rdMolDescriptors.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s), radius=2, nBits=512)
               for s in unique_smiles]

# pairwise Tanimoto distance, lower triangle only (what Butina expects)
dists = []
for i in range(1, len(cluster_fps)):
    sims = DataStructs.BulkTanimotoSimilarity(cluster_fps[i], cluster_fps[:i])
    dists.extend([1 - s for s in sims])

CLUSTER_CUTOFF = 0.4  # distance cutoff -> ~0.6 Tanimoto similarity threshold
clusters = Butina.ClusterData(dists, len(cluster_fps), CLUSTER_CUTOFF, isDistData=True)

smiles_to_cluster = {}
for cluster_id, member_idxs in enumerate(clusters):
    for idx in member_idxs:
        smiles_to_cluster[unique_smiles[idx]] = cluster_id

df['cluster'] = df['smiles'].map(smiles_to_cluster)

cluster_sizes = df.drop_duplicates('smiles')['cluster'].value_counts()
print(f'{len(clusters):,} clusters from {len(unique_smiles):,} molecules (similarity threshold {1-CLUSTER_CUTOFF})')
print(f'Largest cluster: {cluster_sizes.max()} molecules, singleton clusters: {(cluster_sizes == 1).sum()}')

In [ ]:
kfold_cluster_r2 = {}

for target in TARGET_COLS:
    sub = df.dropna(subset=[target]).reset_index(drop=True)
    X, y, groups = sub[FEATURE_COLS], sub[target], sub['cluster']

    gkf = GroupKFold(n_splits=5)
    fold_scores = []
    for train_idx, test_idx in gkf.split(X, y, groups=groups):
        rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
        rf.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = rf.predict(X.iloc[test_idx])
        fold_scores.append(r2_score(y.iloc[test_idx], pred))

    kfold_cluster_r2[target] = fold_scores
    print(f'{target}: {[round(s, 3) for s in fold_scores]}  '
          f'mean={np.mean(fold_scores):.3f}  std={np.std(fold_scores):.3f}')

**Results vs. scaffold (Section 12) and solvent-held-out (Section 14)**

| Target | Fold scores (cluster) | Mean R² | Std R² | Mean R² (scaffold) | Mean R² (solvent) |
|---|---|---|---|---|---|
| Absorption max | 0.232, 0.609, 0.713, 0.729, 0.798 | 0.616 | 0.201 | 0.641 | 0.890 |
| Emission max | 0.579, 0.508, 0.667, 0.545, 0.596 | 0.579 | 0.053 | 0.626 | 0.824 |
| log(ext coeff) | 0.338, 0.184, 0.512, 0.540, 0.454 | 0.406 | 0.130 | 0.454 | 0.656 |
| Quantum yield | 0.187, 0.193, 0.208, 0.200, 0.250 | 0.207 | 0.022 | 0.242 | 0.539 |

2,444 clusters from 6,865 molecules.

The cluster split is the hardest of the three for every target, slightly below the scaffold split in each case. Unlike the frequency split in Section 11, the direction is consistent across all targets. Grouping by overall similarity is a stricter definition of "unseen" than grouping by scaffold.

Absorption max again has the largest fold-to-fold variation (std 0.201, vs. 0.154 under the scaffold split), with one fold at 0.232. This is a second, independent grouping method showing that absorption max's performance depends strongly on which molecules are held out.

Across all three generalization tests the ordering is the same for every target: solvent held out (easiest) > scaffold > cluster (hardest).

## 17. Final tuned model

The final configuration combines the conclusions of Sections 9–16:

- Scaffold split, since it tests generalization to new chemical structures, which is the question this project addresses. The solvent and cluster splits provide context on either side of it.
- Combined feature set (fingerprint + rule features), which beat fingerprint-only on every target in Section 15.
- k-fold evaluation (Section 12), not a single split.

`GridSearchCV` with the Section 10 grid and scaffold-grouped 5-fold CV selects hyperparameters per target. The chosen hyperparameters are then retrained from scratch on each of the five scaffold folds and the mean ± std of those runs is reported.

In [ ]:
final_results = {}

for target in TARGET_COLS:
    sub = df.dropna(subset=[target]).reset_index(drop=True)
    X, y, groups = sub[FEATURE_SETS['combined']], sub[target], sub['scaffold']

    gkf = GroupKFold(n_splits=5)
    cv_splits = list(gkf.split(X, y, groups=groups))

    rf = RandomForestRegressor(random_state=42, n_jobs=1)
    search = GridSearchCV(rf, PARAM_GRID, cv=cv_splits, scoring='r2', n_jobs=-1)
    search.fit(X, y)

    # retrain the chosen hyperparameters on each fold rather than relying on best_score_
    fold_scores, fold_mae = [], []
    for train_idx, test_idx in gkf.split(X, y, groups=groups):
        rf_final = RandomForestRegressor(**search.best_params_, random_state=42, n_jobs=-1)
        rf_final.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = rf_final.predict(X.iloc[test_idx])
        fold_scores.append(r2_score(y.iloc[test_idx], pred))
        fold_mae.append(mean_absolute_error(y.iloc[test_idx], pred))

    final_results[target] = {
        'best_params': search.best_params_,
        'mean_r2': float(np.mean(fold_scores)),
        'std_r2': float(np.std(fold_scores)),
        'mean_mae': float(np.mean(fold_mae)),
    }
    print(f'[{target}] best_params={search.best_params_}  '
          f'final R2={np.mean(fold_scores):.3f} (std {np.std(fold_scores):.3f})  MAE={np.mean(fold_mae):.3f}')

pd.DataFrame(final_results).T

**Final results and progression from the baseline**

| Target | fp-only, untuned | + rule features, untuned | + tuning (final) | Best hyperparameters |
|---|---|---|---|---|
| Absorption max | 0.641 | 0.659 | 0.670 ± 0.176 | n_estimators=300, max_depth=None, max_features=0.3, min_samples_leaf=1 |
| Emission max | 0.626 | 0.650 | 0.672 ± 0.010 | n_estimators=300, max_depth=None, max_features=0.3, min_samples_leaf=1 |
| log(ext coeff) | 0.454 | 0.476 | 0.521 ± 0.081 | n_estimators=300, max_depth=None, max_features='sqrt', min_samples_leaf=1 |
| Quantum yield | 0.242 | 0.244 | 0.266 ± 0.039 | n_estimators=300, max_depth=None, max_features='sqrt', min_samples_leaf=1 |

All values are scaffold-grouped 5-fold means. Runtime: ~41 minutes.

Each step improved every target. Relative to the untuned fingerprint-only baseline, the final models gain +0.029 (absorption max), +0.046 (emission max), +0.067 (log ext coeff) and +0.024 (quantum yield) in R².

log(ext coeff) gains the most despite having the least data (8,189 rows). With fewer examples the fingerprint covers less of fragment space, which may leave more room for the rule features, especially `conjugation_size`, to contribute.

Emission max is the most stable target (std 0.010). Absorption max remains the most variable (std 0.176), consistent with Sections 9, 12 and 16.

Quantum yield remains the weakest target, as expected from its zero-inflated distribution (Section 5.4).

These four configurations are the final per-target models.